---
title: "Module 2 - Tutorial 2"
subtitle: "Semantic Neighbors, Cross-Entropy, and LLM-Style Evaluation"
number-sections: true
date: "2026-02-01"
date-modified: today
date-format: long
engine: jupyter
format:
  html:
    code-overflow: wrap
    toc: true
    toc-depth: 3
categories: ['M02:', 'Tutorial']
description: "Tutorial for Module 2 focused on semantic text representations over SEC chunks."
execute:
  echo: true
  eval: false
---

## Module Learning Objectives

By the end of Module 2, students should be able to:

1. move from cleaned text to structured representations
2. compare sparse and dense text representations
3. interpret semantic similarity in a business-text setting
4. connect vector semantics to neural classification and LLM-style prediction

## Tutorial Learning Objectives

By the end of this tutorial, you will:

1. load SEC text chunks and inspect their metadata
2. compare sparse TF-IDF similarity with dense embedding similarity
3. interpret where semantic retrieval succeeds and where it overgeneralizes
4. compute simple cross-entropy-style predictions from classifier probabilities

## Why This Tutorial Matters

This tutorial connects three ideas that students often learn separately:

- semantic retrieval over embeddings
- probabilistic classification over labels
- LLM-style prediction over token or class distributions

The goal is to show that these are variations on the same underlying workflow: represent text numerically, score alternatives, and interpret probabilities.

## How This Tutorial Connects

- `M02_Lab1` produces the kind of cleaned chunk-level SEC text used here.
- `M02_Lab2` turns this tutorial into a fully executed lab with diagnostics, classifier evaluation, and error analysis.
- `M02_A` extends the same intuition into a graded workflow with Word2Vec, GloVe, classifier comparison, and embedding geometry.

::: {.callout-note}
The tutorial is intentionally guided. It should help students understand the mechanics without giving them a one-click substitute for the lab or assignment.
:::

## Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, log_loss
from sentence_transformers import SentenceTransformer

## Step 1: Load SEC Chunked Text

In [ ]:
chunks_df = pd.read_csv("../data/assignment01/chunks.csv")
chunks_df = chunks_df[["chunk_id", "company", "item", "chunk_text"]].dropna().head(300)
chunks_df.head()

Focus on what each row represents:

- one chunk is now the unit of analysis
- `item` provides weak business context
- `chunk_text` is what we will represent sparsely and densely

## Step 2: Build a Sparse Baseline with TF-IDF

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 2))
X_tfidf = tfidf.fit_transform(chunks_df["chunk_text"])
X_tfidf.shape

Use this baseline to ask a concrete question: if the query changes wording but not meaning, will lexical overlap be enough?

## Step 3: Build Dense Embeddings

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
X_embed = embed_model.encode(
    chunks_df["chunk_text"].tolist(),
    batch_size=32,
    convert_to_numpy=True,
    normalize_embeddings=True
)
X_embed.shape

Dense embeddings often recover related text even when wording differs. That is the practical value of vector semantics.

## Step 4: Compare Sparse and Dense Retrieval

In [ ]:
query = "cybersecurity risk and customer data privacy issues"

q_tfidf = tfidf.transform([query])
scores_tfidf = cosine_similarity(q_tfidf, X_tfidf)[0]

q_embed = embed_model.encode([query], normalize_embeddings=True)
scores_embed = cosine_similarity(q_embed, X_embed)[0]

top_sparse = chunks_df.iloc[np.argsort(scores_tfidf)[-5:][::-1]].copy()
top_dense = chunks_df.iloc[np.argsort(scores_embed)[-5:][::-1]].copy()

top_sparse[["chunk_id", "company", "item"]]
top_dense[["chunk_id", "company", "item"]]

## Step 5: Interpret the Difference

Questions for students:

1. Which method retrieves passages with stronger lexical overlap?
2. Which method retrieves passages that are conceptually similar but worded differently?
3. Where does the dense model become too broad and retrieve generic “risk” language?

## Step 6: Build a Simple Binary Classifier

Create a rough label to mimic a downstream analytics task.

In [ ]:
chunks_df["label"] = chunks_df["chunk_text"].str.contains(
    r"privacy|cyber|security|data breach", case=False, regex=True
).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, chunks_df["label"], test_size=0.25, random_state=42, stratify=chunks_df["label"]
)

clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)
probs = clf.predict_proba(X_test)
preds = clf.predict(X_test)

print(classification_report(y_test, preds))

## Step 7: Connect to Cross-Entropy

In [ ]:
loss = log_loss(y_test, probs)
print("Average cross-entropy / log loss:", round(loss, 4))

Interpretation:

- each prediction is a probability distribution over labels
- log loss penalizes confident mistakes strongly
- this is the same family of objective used when LLMs predict the next token

## Step 8: Inspect High-Loss Examples

In [ ]:
test_rows = chunks_df.iloc[y_test.index].copy()
test_rows["p_positive"] = probs[:, 1]
test_rows["loss_if_true"] = -(
    y_test.values * np.log(np.clip(probs[:, 1], 1e-9, 1.0)) +
    (1 - y_test.values) * np.log(np.clip(probs[:, 0], 1e-9, 1.0))
)

test_rows.sort_values("loss_if_true", ascending=False)[
    ["chunk_id", "company", "item", "p_positive", "loss_if_true"]
].head(10)

This step is the most important diagnostic step in the tutorial. It forces students to read model failures instead of reporting metrics only.

## What Students Should Notice

- TF-IDF is strong when wording aligns with the query.
- Dense embeddings recover semantic neighbors under paraphrase.
- Cross-entropy is not abstract; it is a numeric penalty for bad probability assignments.
- The same probability logic underlies classifiers, embedding rerankers, and LLM next-token prediction.

## Extension Ideas

If time permits, students can:

1. compare a dense classifier against the TF-IDF baseline
2. cluster dense embeddings to identify disclosure themes
3. evaluate whether retrieval quality changes by SEC item type
4. test whether mislabeled or ambiguous chunks produce the largest losses

## AI Use and Share Links {.unnumbered}

If generative AI materially supports your work for this tutorial, include an AI disclosure appendix or separate AI disclosure document if your instructor requests one. Include the complete prompt(s), relevant output excerpt(s), validation steps, and direct shared chat link(s) when available.

![](../M0/M0_lecture01_figures/chatgpt-share.jpg){width="80%" fig-align="center"}

![](../M0/M0_lecture01_figures/gemini-share-conversation.png){width="80%" fig-align="center"}

When possible, use **one continuous chat** for the activity so the full reasoning trail can be reviewed in one place. If you used multiple chats, share **all chats directly related** to the work.

Tools such as **ChatGPT**, **Claude**, and **Gemini** support direct chat sharing. If sharing or export is not available, include copied prompt/output evidence or screenshots instead. Because shared links may be viewable by anyone with the link, do not include confidential, personal, or restricted information in those chats.